# 🎬 OpenSource Clipping — .env Paste Config

Cara pakai:
1. Run **Cell 1 (Setup)** sekali.
2. Paste config `.env` ke **Cell 2**, lalu run.
3. Run **Cell 3 (Validate)**.
4. Run **Cell 4 (Run Pipeline)**.

Contoh isi Cell 2:
```env
VIDEO_URL=https://www.youtube.com/watch?v=Dc4_aBFAYWE
OLLAMA_API_KEY=sk-xxxxxxxx
MAX_CLIPS=5
ASPECT_RATIO=9:16
```

## ① Setup — Install dependencies

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — SETUP
# ═══════════════════════════════════════════════════════════════
import os, subprocess, sys, shutil, re
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    pass

REPO_URL = "https://github.com/troublescope/Ashortai.git"
CLONE_DIR = "/content/opensource-clipping"
Path(CLONE_DIR).mkdir(parents=True, exist_ok=True)

if os.path.isdir(f"{CLONE_DIR}/.git"):
    subprocess.run(["git", "-C", CLONE_DIR, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CLONE_DIR], check=True)

os.chdir(CLONE_DIR)

print("⏳ Installing dependencies…")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# ── Backward-compat patch: GOOGLE_API_KEY only required for Gemini provider ──
main_py = Path(CLONE_DIR) / "main.py"
main_text = main_py.read_text(encoding="utf-8")
old_check = '    if not cfg.api_key_gemini:'
new_check = '    if cfg.ai_provider == "gemini" and not cfg.api_key_gemini:'
if old_check in main_text and new_check not in main_text:
    main_py.write_text(main_text.replace(old_check, new_check), encoding="utf-8")
    print("🩹 Patched main.py: GOOGLE_API_KEY only required when provider is gemini.")
print("✅ Dependencies installed.")

try:
    DRIVE_CACHE = Path("/content/drive/MyDrive/osc_cache")
    DRIVE_CACHE.mkdir(parents=True, exist_ok=True)
    for m in ["blaze_face_full_range.tflite", "face_yolov8m.pt", "face_yolov8n.pt", "face_yolov8s.pt"]:
        src, dst = DRIVE_CACHE / m, Path(CLONE_DIR) / m
        if src.exists() and not dst.exists():
            shutil.copy2(src, dst)
            print(f"📥 Restored {m}")
except Exception:
    pass

print("\n✅ Setup complete.")

## ② Paste .env Config

Paste konfigurasi `.env` di bawah ini, lalu run cell ini.
Semua setting yang tidak diisi akan pakai default T4-optimized.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — PASTE .ENV CONFIG DI BAWAH INI
# ═══════════════════════════════════════════════════════════════
# Ganti isi triple-quote di bawah dengan config .env kamu
ENV_CONFIG = """
# paste config .env kamu di sini
"""

# ── DEFAULT CONFIG (T4-optimized, jangan hapus) ──
DEFAULTS = {
    "VIDEO_URL": "",
    "SOURCE_PLATFORM": "youtube",
    "AI_PROVIDER": "ollama",
    "AI_MODEL": "gemini-3-flash-preview:cloud",
    "OLLAMA_URL": "https://ollama.com",
    "OLLAMA_FALLBACK_URL": "https://ollama.com",
    "OLLAMA_FALLBACK_MODEL": "gemini-3-flash-preview:cloud",
    "GEMINI_MODEL": "gemini-3-flash-preview",
    "OLLAMA_API_KEY": "",
    "GOOGLE_API_KEY": "",
    "NVIDIA_API_KEY": "",
    "MAX_CLIPS": "5",
    "ASPECT_RATIO": "9:16",
    "RENDER_HEIGHT": "1080",
    "SOURCE_HEIGHT": "max",
    "FONT_STYLE": "HORMOZI",
    "FACE_DETECTOR": "mediapipe",
    "YOLO_SIZE": "8m",
    "ENABLE_SUBTITLES": "True",
    "USE_KARAOKE_EFFECT": "True",
    "WORDS_PER_SUBTITLE": "5",
    "USE_YT_TRANSCRIPT_API": "True",
    "USE_DLP_SUBS": "True",
    "WHISPER_MODEL": "large-v3",
    "WHISPER_DEVICE": "auto",
    "WHISPER_COMPUTE_TYPE": "float16",
    "VIDEO_PRESET": "ultrafast",
    "VIDEO_SCALE_ALGO": "bicubic",
    "VIDEO_CQ": "23",
    "VIDEO_CRF": "20",
    "VIDEO_SHARPEN": "False",
    "ENABLE_COLAB_CLEANUP": "True",
    "DISABLE_BGM": "True",
    "DISABLE_BROLL": "True",
    "DISABLE_HOOK_GLITCH": "False",
    "ENABLE_HOOK_V2": "False",
    "HOOK_DURATION": "3",
    "ENABLE_SPLIT_SCREEN": "False",
    "SPLIT_TRIGGER": "diarization",
    "ENABLE_CAMERA_SWITCH": "False",
    "DISABLE_SEGMENT_TRIM": "False",
    "AGGRESSIVE_SILENCE_TRIM": "False",
    "HF_TOKEN": "",
}

def parse_env(text: str) -> dict:
    result = {}
    for raw in text.strip().splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        if "=" not in line:
            continue
        key, _, value = line.partition("=")
        result[key.strip()] = value.strip()
    return result

user_cfg = parse_env(ENV_CONFIG)
cfg = dict(DEFAULTS)
cfg.update(user_cfg)

# ── Convert typed values ──
def to_bool(v):
    return str(v).strip().lower() in ("true", "1", "yes", "on")

def to_int(v, default=0):
    try:
        return int(str(v).strip())
    except Exception:
        return default

VIDEO_URL = cfg["VIDEO_URL"]
SOURCE_PLATFORM = cfg["SOURCE_PLATFORM"]
AI_PROVIDER = cfg["AI_PROVIDER"]
AI_MODEL = cfg["AI_MODEL"]
OLLAMA_URL = cfg["OLLAMA_URL"]
OLLAMA_FALLBACK_URL = cfg["OLLAMA_FALLBACK_URL"]
OLLAMA_FALLBACK_MODEL = cfg["OLLAMA_FALLBACK_MODEL"]
GEMINI_MODEL = cfg["GEMINI_MODEL"]
OLLAMA_API_KEY = cfg["OLLAMA_API_KEY"]
GOOGLE_API_KEY = cfg["GOOGLE_API_KEY"]
NVIDIA_API_KEY = cfg["NVIDIA_API_KEY"]
MAX_CLIPS = to_int(cfg["MAX_CLIPS"], 5)
ASPECT_RATIO = cfg["ASPECT_RATIO"]
RENDER_HEIGHT = cfg["RENDER_HEIGHT"]
SOURCE_HEIGHT = cfg["SOURCE_HEIGHT"]
FONT_STYLE = cfg["FONT_STYLE"]
FACE_DETECTOR = cfg["FACE_DETECTOR"]
YOLO_SIZE = cfg["YOLO_SIZE"]
ENABLE_SUBTITLES = to_bool(cfg["ENABLE_SUBTITLES"])
USE_KARAOKE_EFFECT = to_bool(cfg["USE_KARAOKE_EFFECT"])
WORDS_PER_SUBTITLE = to_int(cfg["WORDS_PER_SUBTITLE"], 5)
USE_YT_TRANSCRIPT_API = to_bool(cfg["USE_YT_TRANSCRIPT_API"])
USE_DLP_SUBS = to_bool(cfg["USE_DLP_SUBS"])
WHISPER_MODEL = cfg["WHISPER_MODEL"]
WHISPER_DEVICE = cfg["WHISPER_DEVICE"]
WHISPER_COMPUTE_TYPE = cfg["WHISPER_COMPUTE_TYPE"]
VIDEO_PRESET = cfg["VIDEO_PRESET"]
VIDEO_SCALE_ALGO = cfg["VIDEO_SCALE_ALGO"]
VIDEO_CQ = to_int(cfg["VIDEO_CQ"], 23)
VIDEO_CRF = to_int(cfg["VIDEO_CRF"], 20)
VIDEO_SHARPEN = to_bool(cfg["VIDEO_SHARPEN"])
ENABLE_COLAB_CLEANUP = to_bool(cfg["ENABLE_COLAB_CLEANUP"])
DISABLE_BGM = to_bool(cfg["DISABLE_BGM"])
DISABLE_BROLL = to_bool(cfg["DISABLE_BROLL"])
DISABLE_HOOK_GLITCH = to_bool(cfg["DISABLE_HOOK_GLITCH"])
ENABLE_HOOK_V2 = to_bool(cfg["ENABLE_HOOK_V2"])
HOOK_DURATION = to_int(cfg["HOOK_DURATION"], 3)
ENABLE_SPLIT_SCREEN = to_bool(cfg["ENABLE_SPLIT_SCREEN"])
SPLIT_TRIGGER = cfg["SPLIT_TRIGGER"]
ENABLE_CAMERA_SWITCH = to_bool(cfg["ENABLE_CAMERA_SWITCH"])
DISABLE_SEGMENT_TRIM = to_bool(cfg["DISABLE_SEGMENT_TRIM"])
AGGRESSIVE_SILENCE_TRIM = to_bool(cfg["AGGRESSIVE_SILENCE_TRIM"])
HF_TOKEN = cfg["HF_TOKEN"]

print("✅ Config loaded. Overridden keys:")
for k in sorted(user_cfg.keys()):
    v = user_cfg[k]
    display = v if "KEY" not in k.upper() else "***"
    print(f"  • {k} = {display}")
if not user_cfg:
    print("  (using all defaults)")

## ✅ Validate

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — VALIDATION
# ═══════════════════════════════════════════════════════════════
errors = []
warnings = []

if not VIDEO_URL.strip():
    errors.append("❌ VIDEO_URL wajib diisi.")
elif not VIDEO_URL.strip().startswith(("http://", "https://")):
    errors.append("❌ VIDEO_URL harus diawali http:// atau https://.")

if AI_PROVIDER == "ollama" and OLLAMA_URL.startswith("https://ollama.com") and not OLLAMA_API_KEY.strip():
    errors.append("❌ OLLAMA_API_KEY wajib untuk Ollama Cloud.")
if AI_PROVIDER == "gemini" and not GOOGLE_API_KEY.strip():
    errors.append("❌ GOOGLE_API_KEY wajib jika AI_PROVIDER='gemini'.")
if AI_PROVIDER == "nvidia" and not NVIDIA_API_KEY.strip():
    errors.append("❌ NVIDIA_API_KEY wajib jika AI_PROVIDER='nvidia'.")

if AI_PROVIDER != "gemini" and not GOOGLE_API_KEY.strip():
    warnings.append("⚠️ GOOGLE_API_KEY kosong. Fallback ke Gemini tidak akan bekerja.")

if not (1 <= MAX_CLIPS <= 10):
    errors.append("❌ MAX_CLIPS harus 1-10.")
if not (1 <= WORDS_PER_SUBTITLE <= 20):
    errors.append("❌ WORDS_PER_SUBTITLE harus 1-20.")
if not (1 <= HOOK_DURATION <= 10):
    errors.append("❌ HOOK_DURATION harus 1-10 detik.")
if not (15 <= VIDEO_CQ <= 50):
    errors.append("❌ VIDEO_CQ harus 15-50.")
if not (15 <= VIDEO_CRF <= 50):
    errors.append("❌ VIDEO_CRF harus 15-50.")

if ASPECT_RATIO != "9:16" and (ENABLE_SPLIT_SCREEN or ENABLE_CAMERA_SWITCH):
    warnings.append("⚠️ Split-screen/camera-switch paling cocok untuk 9:16.")
if (ENABLE_SPLIT_SCREEN or ENABLE_CAMERA_SWITCH) and not HF_TOKEN.strip():
    errors.append("❌ HF_TOKEN wajib untuk split-screen atau camera-switch.")

print("═" * 60)
print("📋 SETTINGS")
print("═" * 60)
print(f"  Video URL           : {VIDEO_URL or '—'}")
print(f"  Source Platform     : {SOURCE_PLATFORM}")
print(f"  AI Provider         : {AI_PROVIDER}")
print(f"  AI Model            : {AI_MODEL}")
print(f"  Ollama API Key      : {'✅ set' if OLLAMA_API_KEY.strip() else '—'}")
print(f"  Google API Key      : {'✅ set' if GOOGLE_API_KEY.strip() else '—'}")
print(f"  NVIDIA API Key      : {'✅ set' if NVIDIA_API_KEY.strip() else '—'}")
print(f"  Max Clips           : {MAX_CLIPS}")
print(f"  Aspect Ratio        : {ASPECT_RATIO}")
print(f"  Render Height       : {RENDER_HEIGHT}")
print(f"  Subtitles           : {ENABLE_SUBTITLES}")
print(f"  YouTube Transcript  : {USE_YT_TRANSCRIPT_API}")
print(f"  Colab Cleanup       : {ENABLE_COLAB_CLEANUP}")

if warnings:
    print("\n⚠️ WARNINGS")
    for w in warnings:
        print(f"  {w}")

if errors:
    print("\n❌ ERRORS")
    for e in errors:
        print(f"  {e}")
    raise SystemExit("Validation failed.")

print("\n✅ Ready to run!")

## 🚀 Run Pipeline

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — RUN PIPELINE
# ═══════════════════════════════════════════════════════════════
import subprocess, sys, shutil, time
from datetime import datetime
from pathlib import Path

# ── Write .env ──
env_lines = []
if OLLAMA_API_KEY.strip(): env_lines.append(f"OLLAMA_API_KEY={OLLAMA_API_KEY.strip()}")
if GOOGLE_API_KEY.strip(): env_lines.append(f"GOOGLE_API_KEY={GOOGLE_API_KEY.strip()}")
if NVIDIA_API_KEY.strip(): env_lines.append(f"NVIDIA_API_KEY={NVIDIA_API_KEY.strip()}")
if HF_TOKEN.strip(): env_lines.append(f"HF_TOKEN={HF_TOKEN.strip()}")
if env_lines:
    Path(CLONE_DIR, ".env").write_text("\n".join(env_lines) + "\n", encoding="utf-8")
    print("🔐 .env written.")

url = VIDEO_URL.strip()

# ── Unique output folder ──
def _slugify(s: str) -> str:
    s = s.split("?")[0].split("/")[-1]
    s = re.sub(r"[^\w\-_. ]", "", s).strip()
    return s[:40] or "clip"

output_slug = f"{_slugify(url)}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUTPUT_DIR = Path(CLONE_DIR) / "outputs" / output_slug
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"📁 Output target: {OUTPUT_DIR}")

cmd = [
    sys.executable, "main.py",
    "--url", url,
    "--source", SOURCE_PLATFORM,
    "--clips", str(MAX_CLIPS),
    "--ratio", ASPECT_RATIO,
    "--render-height", str(RENDER_HEIGHT),
    "--source-height", str(SOURCE_HEIGHT),
    "--font-style", FONT_STYLE,
    "--face-detector", FACE_DETECTOR,
    "--yolo-size", YOLO_SIZE,
    "--ai-provider", AI_PROVIDER,
    "--gemini-model", GEMINI_MODEL,
    "--ollama-url", OLLAMA_URL,
    "--ollama-model", AI_MODEL,
    "--ollama-fallback-url", OLLAMA_FALLBACK_URL,
    "--ollama-fallback-model", OLLAMA_FALLBACK_MODEL,
    "--whisper-model", WHISPER_MODEL,
    "--whisper-device", WHISPER_DEVICE,
    "--whisper-compute-type", WHISPER_COMPUTE_TYPE,
    "--video-preset", VIDEO_PRESET,
    "--video-scale-algo", VIDEO_SCALE_ALGO,
    "--video-cq", str(VIDEO_CQ),
    "--video-crf", str(VIDEO_CRF),
    "--words-per-sub", str(WORDS_PER_SUBTITLE),
    "--hook-duration", str(HOOK_DURATION),
]

def add_flag(flag, cond):
    if cond:
        cmd.append(flag)

add_flag("--colab-cleanup", ENABLE_COLAB_CLEANUP)
add_flag("--use-yt-transcript-api", USE_YT_TRANSCRIPT_API)
add_flag("--use-dlp-subs", USE_DLP_SUBS)
add_flag("--no-bgm", DISABLE_BGM)
add_flag("--no-broll", DISABLE_BROLL)
add_flag("--no-subs", not ENABLE_SUBTITLES)
add_flag("--no-karaoke", not USE_KARAOKE_EFFECT)
add_flag("--no-hook", DISABLE_HOOK_GLITCH)
add_flag("--camera-switch", ENABLE_CAMERA_SWITCH)
add_flag("--hook-v2", ENABLE_HOOK_V2)
add_flag("--no-segment-trim", DISABLE_SEGMENT_TRIM)
add_flag("--silence-trim", AGGRESSIVE_SILENCE_TRIM)
add_flag("--video-sharpen", VIDEO_SHARPEN)
add_flag("--split-screen", ENABLE_SPLIT_SCREEN)

if ENABLE_SPLIT_SCREEN:
    cmd += ["--split-trigger", SPLIT_TRIGGER]

print("🚀 Starting pipeline…")
print(f"URL: {url}")
print("Command:", " ".join(cmd))
print("─" * 60)

start = time.time()
result = subprocess.run(cmd, cwd=CLONE_DIR)
if result.returncode != 0:
    print(f"❌ Pipeline exited with code {result.returncode}")
elapsed = time.time() - start

print("─" * 60)
print(f"⏱️ Total time: {elapsed/60:.1f} minutes")

# ── Collect outputs into unique folder ──
base_out = Path(CLONE_DIR) / "outputs"
if base_out.exists():
    moved = 0
    for f in sorted(base_out.iterdir()):
        if f.is_file():
            try:
                shutil.move(str(f), str(OUTPUT_DIR / f.name))
                moved += 1
            except Exception as e:
                print(f"⚠️ Could not move {f.name}: {e}")
    print(f"\n📁 {moved} file(s) moved to: {OUTPUT_DIR}")

    files = sorted(OUTPUT_DIR.glob("*_ready.mp4")) + sorted(OUTPUT_DIR.glob("*.jpg"))
    if files:
        print(f"\n📦 Output files ({len(files)}):")
        for f in files:
            print(f"   • {f.name} ({f.stat().st_size/1024/1024:.1f} MB)")
    if list(OUTPUT_DIR.iterdir()):
        zip_path = OUTPUT_DIR / "outputs.zip"
        shutil.make_archive(str(zip_path.with_suffix("")), "zip", root_dir=str(OUTPUT_DIR))
        print(f"\n🗜️ outputs.zip → {zip_path.stat().st_size/1024/1024:.1f} MB")
        try:
            from google.colab import files
            files.download(str(zip_path))
        except Exception:
            pass
else:
    print("⚠️ outputs/ directory not found.")